# Engineered Features


This notebook collects functions that adds features to our rat sightings dataframe. Since these functions are reused throughout the project, we have written them here to demonstrate their usage and provided explanations and examples as needed.

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

We demonstrate the functions we will use on daily rat sightings data.

In [26]:
# we import the data and clean it for future use
rs = pd.read_csv('../data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs.head(5)

,ds,y
0,2020-01-01,17
1,2020-01-02,40
2,2020-01-03,41
3,2020-01-04,25
4,2020-01-05,17


At the moment, the dataframe is quite bare. It has the number of rats spotted per day and the date.

For our functions, we first need to set the index as the date and ensure it is in datetime format.

In [27]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)

Our first function adds the day of the week, quarter, month, year, day of the year, day of the month, and week of the year as features to consider.

In [28]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

rs = create_features(rs)
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear
ds,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1
2020-01-02,40,3,1,1,2020,2,2,1
2020-01-03,41,4,1,1,2020,3,3,1
2020-01-04,25,5,1,1,2020,4,4,1
2020-01-05,17,6,1,1,2020,5,5,1
...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9
2025-02-25,61,1,1,2,2025,56,25,9
2025-02-26,61,2,1,2,2025,57,26,9


The add_cyclic() function adds Fourier terms which is meant to help XGBoost try to capture seasonal behaviors.

In [29]:
def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

rs = add_cyclic(rs)

rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,month_sin,month_cos
ds,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,0.500000,0.866025
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,0.500000,0.866025
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,0.500000,0.866025
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,0.500000,0.866025
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,0.500000,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,0.866025,0.500000
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,0.866025,0.500000
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,0.866025,0.500000


The next two functions do essentially do the same thing. One just captures short term lags while the other adds longer term lags to capture seasonality.

We always make sure to use at least 14 days for our lags so that our forecasts will not use information we don't know yet.

In [30]:
def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    return df


def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df


rs = add_lags(rs)
rs = add_seasonal_lags(rs)

rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,lag362,lag363,lag364,lag365,lag366,lag367,lag730,lag1095,lag1460,lag1825
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,52.0,52.0,52.0,52.0,52.0,52.0,45.0,45.0,69.0,43.0
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,63.0,63.0,63.0,63.0,63.0,63.0,42.0,38.0,54.0,50.0
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,78.0,78.0,78.0,78.0,78.0,78.0,72.0,50.0,50.0,29.0


We can also consider moving averages shifted by 14 days so we do not use future information in our forecasts. The idea here would be to smooth out the predictions since rat sightings have a tendency to spike or dip massively.

In [31]:
def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    # Must shift by 14 days because we do not want to let there be temporal leakage in our evaluations
    df['ma7'] = df['y'].shift(14).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(14).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(14).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(14).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(14).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(14).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(14).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(14).rolling(window=365).mean()
    
    return df

rs = add_moving_averages(rs)
rs


,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,lag1460,lag1825,ma7,ma30,ma60,ma90,ma120,ma150,ma180,ma365
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,69.0,43.0,42.571429,44.833333,39.450000,40.911111,45.866667,52.213333,57.788889,67.054795
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,54.0,50.0,43.428571,45.166667,39.483333,40.600000,45.650000,52.146667,57.477778,66.961644
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,50.0,29.0,41.142857,44.533333,39.883333,40.477778,45.550000,51.953333,57.300000,66.931507


We import some weather data.

In [32]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

We add the imported weather data as features. All of the weather data here is measured in celsius and precipitation and snowfall is measured in millimeters (mm) and centimeters (cm) respectively.

In [33]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Remove overlapping columns to avoid join errors
    overlap = wd.columns.intersection(df.columns)
    wd = wd.drop(columns=overlap)
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

rs = add_weather_data(rs,wd)
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,ma180,ma365,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,precipitation_sum,snowfall_sum
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,4.4,-0.4,2.2,-0.3,-5.3,-3.4,0.0,0.0
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,8.7,-2.1,2.5,4.5,-6.6,-1.7,0.0,0.0
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,9.4,2.2,6.6,7.3,-1.1,4.1,4.3,0.0
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,9.6,5.8,7.7,7.9,1.0,5.7,6.0,0.0
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,5.1,-1.2,2.7,-1.0,-5.0,-3.5,0.8,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,57.788889,67.054795,8.1,-1.5,3.4,4.0,-4.6,-0.4,0.0,0.0
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,57.477778,66.961644,12.4,3.5,7.0,9.8,0.1,3.9,0.0,0.0
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,57.300000,66.931507,12.9,2.2,6.7,8.4,-1.4,3.3,0.1,0.0


We also add lagged weather features.

In [35]:
def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

rs = add_more_weather_feature(rs)
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,apparent_temperature_min_lag240,apparent_temperature_min_lag270,apparent_temperature_min_lag300,apparent_temperature_min_lag330,apparent_temperature_min_lag360,apparent_temperature_min_lag365,apparent_temperature_min_lag730,temperature_2m_max_lag14,temperature_2m_max_lag30,temperature_2m_max_lag60
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,18.2,12.8,8.5,0.5,-9.1,-11.1,-10.8,0.2,-2.2,-0.2
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,22.2,10.2,8.7,2.2,-1.5,-6.9,-8.1,-0.4,4.1,2.5
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,15.2,11.2,8.7,2.0,3.4,-4.1,-7.1,0.6,2.0,7.5


We also add differences weather data to account for changes in climate year over year.

In [36]:
def add_yearly_temp_diffs(df, horizon_days=14, years=[1,2,3,4]):
    df = df.copy()

    # Ensure datetime index
    df.index = pd.to_datetime(df.index)

    # Base shifted temps (outside forecast horizon)
    base_min = df['apparent_temperature_min'].shift(horizon_days)
    base_max = df['temperature_2m_max'].shift(horizon_days)

    for y in years:
        lag_days = 365 * y

        df[f'apparent_temperature_min_diff_{y}y'] = (
            base_min - df['apparent_temperature_min'].shift(horizon_days + lag_days)
        )

        df[f'temperature_2m_max_diff_{y}y'] = (
            base_max - df['temperature_2m_max'].shift(horizon_days + lag_days)
        )

    return df

rs = add_yearly_temp_diffs(rs)
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,temperature_2m_max_lag30,temperature_2m_max_lag60,apparent_temperature_min_diff_1y,temperature_2m_max_diff_1y,apparent_temperature_min_diff_2y,temperature_2m_max_diff_2y,apparent_temperature_min_diff_3y,temperature_2m_max_diff_3y,apparent_temperature_min_diff_4y,temperature_2m_max_diff_4y
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,-2.2,-0.2,-9.7,-8.7,-5.0,-6.5,-2.4,-11.7,-0.6,-0.4
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,4.1,2.5,-7.9,-9.6,-8.5,-7.3,-14.5,-13.6,3.1,2.1
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,2.0,7.5,-0.3,-3.2,-2.4,-12.3,2.6,-4.5,7.5,3.1


We one-hot encode federal holidays as a feature. Of course, certain federal holidays have a bigger impact on human behavior and this will not capture it.

In [37]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

rs = add_federal_holidays(rs)
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,temperature_2m_max_lag60,apparent_temperature_min_diff_1y,temperature_2m_max_diff_1y,apparent_temperature_min_diff_2y,temperature_2m_max_diff_2y,apparent_temperature_min_diff_3y,temperature_2m_max_diff_3y,apparent_temperature_min_diff_4y,temperature_2m_max_diff_4y,is_federal_holiday
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,-0.2,-9.7,-8.7,-5.0,-6.5,-2.4,-11.7,-0.6,-0.4,0
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,2.5,-7.9,-9.6,-8.5,-7.3,-14.5,-13.6,3.1,2.1,0
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,7.5,-0.3,-3.2,-2.4,-12.3,2.6,-4.5,7.5,3.1,0


We one-hot encode days where a new law/city policy has been implemented. For example, we keep track of when new trash laws/policies were put in place and when the rat mitigation zones in NYC were implemented.

In [38]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df



rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,temperature_2m_max_diff_2y,apparent_temperature_min_diff_3y,temperature_2m_max_diff_3y,apparent_temperature_min_diff_4y,temperature_2m_max_diff_4y,is_federal_holiday,Trash_Law,New_Trash_Law,Rat_Mitigation_Zone,Rat_Czar_Appointed
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,0,1,2,2025,55,24,9,0.000000,1.000000,...,-6.5,-2.4,-11.7,-0.6,-0.4,0,1,1,1,1
2025-02-25,61,1,1,2,2025,56,25,9,0.781831,0.623490,...,-7.3,-14.5,-13.6,3.1,2.1,0,1,1,1,1
2025-02-26,61,2,1,2,2025,57,26,9,0.974928,-0.222521,...,-12.3,2.6,-4.5,7.5,3.1,0,1,1,1,1
